# Selection Efficiency Figure

This notebook stays thin on purpose:
- it only reads the exported balanced-draw artifacts from `results/.../analysis/ping_pong_balanced_draw_stats_inference`
- it pools short and long variants within each cohort
- it reproduces the two-panel selection-efficiency figure from those artifacts


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'xtick.labelsize': 7.5,
    'ytick.labelsize': 7.5,
    'legend.fontsize': 7.5,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
    'axes.linewidth': 0.6,
    'xtick.major.width': 0.5,
    'ytick.major.width': 0.5,
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'lines.linewidth': 1.3,
    'lines.markersize': 4,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})


In [ ]:
root_candidates = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next(
    (
        candidate.resolve()
        for candidate in root_candidates
        if (candidate / 'results').exists() and (candidate / 'notebooks').exists()
    ),
    Path.cwd().resolve(),
)
print(f'Using repo root: {REPO_ROOT}')

ANALYSIS_SUBDIR = 'analysis/ping_pong_balanced_draw_stats_inference'

VARIANTS = {
    'short': {
        'label': 'MIMIC-IV short 0-shot',
        'cohort': 'MIMIC-IV',
        'output_root': REPO_ROOT / 'results/mimic_streamlined_pipeline_small_models_upd_short',
    },
    'long': {
        'label': 'MIMIC-IV long 0-shot',
        'cohort': 'MIMIC-IV',
        'output_root': REPO_ROOT / 'results/mimic_streamlined_pipeline_small_models_upd_long',
    },
    'indic_short': {
        'label': 'Indic short 0-shot',
        'cohort': 'Indian OCR',
        'output_root': REPO_ROOT / 'results/mimic_streamlined_pipeline_small_models_indic_upd_short',
    },
    'indic_long': {
        'label': 'Indic long 0-shot',
        'cohort': 'Indian OCR',
        'output_root': REPO_ROOT / 'results/mimic_streamlined_pipeline_small_models_indic_upd_long',
    },
}

COHORTS = {
    'MIMIC-IV': {
        'title': 'MIMIC-IV ($n=628$)',
        'variants': ['short', 'long'],
    },
    'Indian OCR': {
        'title': 'Indian OCR ($n=340$)',
        'variants': ['indic_short', 'indic_long'],
    },
}

REGIME_ORDER = [
    'pure_random',
    'reveal_one_by_one_yhat_round_robin',
    'reveal_one_by_one_yhat_round_robin_prob_no_update',
    'reveal_one_by_one_yhat_round_robin_prob_update',
]

REGIME_PRETTY_NAMES = {
    'pure_random': 'Random',
    'reveal_one_by_one_yhat_round_robin': '$\hat{y}$-Bucket',
    'reveal_one_by_one_yhat_round_robin_prob_no_update': 'NoUpdate',
    'reveal_one_by_one_yhat_round_robin_prob_update': 'WithUpdate',
}

REGIME_COLORS = {
    'pure_random': '#999999',
    'reveal_one_by_one_yhat_round_robin': '#E69F00',
    'reveal_one_by_one_yhat_round_robin_prob_no_update': '#56B4E9',
    'reveal_one_by_one_yhat_round_robin_prob_update': '#D55E00',
}

REGIME_LINESTYLES = {
    'pure_random': '--',
    'reveal_one_by_one_yhat_round_robin': '-.',
    'reveal_one_by_one_yhat_round_robin_prob_no_update': ':',
    'reveal_one_by_one_yhat_round_robin_prob_update': '-',
}

REGIME_MARKERS = {
    'pure_random': 's',
    'reveal_one_by_one_yhat_round_robin': '^',
    'reveal_one_by_one_yhat_round_robin_prob_no_update': 'D',
    'reveal_one_by_one_yhat_round_robin_prob_update': 'o',
}


In [ ]:
ARTIFACTS = {}
required_files = {
    'agg_k': 'balanced_draw_stats_agg_k.csv',
    'primary': 'balanced_draw_stats_primary_mean_over_k_comparisons.csv',
    'per_k': 'balanced_draw_stats_per_k_wilcoxon_comparisons.csv',
    'model': 'balanced_draw_stats_model_robustness_mean_over_k.csv',
    'regime_summary': 'balanced_draw_stats_regime_full_range_summary.csv',
    'metadata': 'inference_metadata.json',
}

for variant, meta in VARIANTS.items():
    analysis_dir = meta['output_root'] / ANALYSIS_SUBDIR
    missing = [
        name for name, filename in required_files.items()
        if not (analysis_dir / filename).exists()
    ]
    if missing:
        raise FileNotFoundError(f'{variant}: missing {missing} under {analysis_dir}')

    ARTIFACTS[variant] = {
        'analysis_dir': analysis_dir,
        'agg_k': pd.read_csv(analysis_dir / required_files['agg_k']),
        'primary': pd.read_csv(analysis_dir / required_files['primary']),
        'per_k': pd.read_csv(analysis_dir / required_files['per_k']),
        'model': pd.read_csv(analysis_dir / required_files['model']),
        'regime_summary': pd.read_csv(analysis_dir / required_files['regime_summary']),
        'metadata': json.loads((analysis_dir / required_files['metadata']).read_text()),
    }
    print(f'{variant}: {analysis_dir}')

display(
    pd.concat(
        [
            pd.DataFrame([
                {
                    'variant': variant,
                    'cohort': VARIANTS[variant]['cohort'],
                    'analysis_dir': str(payload['analysis_dir']),
                    'k_rows': len(payload['agg_k']),
                    'regime_summary_rows': len(payload['regime_summary']),
                    'k_values': ', '.join(str(int(x)) for x in payload['metadata']['k_values']),
                }
            ])
            for variant, payload in ARTIFACTS.items()
        ],
        ignore_index=True,
    )
)


In [ ]:
pooled_agg_rows = []
pooled_summary_rows = []

for cohort, meta in COHORTS.items():
    agg_df = pd.concat(
        [ARTIFACTS[variant]['agg_k'].assign(variant=variant) for variant in meta['variants']],
        ignore_index=True,
    )
    pooled_agg = (
        agg_df.groupby(['regime', 'k'], as_index=False)
        .agg(
            global_selected_over_annotated_ratio=('global_selected_over_annotated_ratio', 'mean'),
            completed_rate=('completed_rate', 'mean'),
        )
        .assign(cohort=cohort)
    )
    pooled_agg_rows.append(pooled_agg)

    regime_summary_df = pd.concat(
        [ARTIFACTS[variant]['regime_summary'].assign(variant=variant) for variant in meta['variants']],
        ignore_index=True,
    )
    pooled_regime_summary = (
        regime_summary_df.groupby('regime', as_index=False)
        .agg(
            mean_ratio_over_k=('mean_ratio_over_k', 'mean'),
            median_ratio_over_k=('median_ratio_over_k', 'mean'),
            mean_normalized_auc_ratio=('mean_normalized_auc_ratio', 'mean'),
            median_normalized_auc_ratio=('median_normalized_auc_ratio', 'mean'),
            n_pairs=('n_pairs', 'sum'),
        )
        .assign(cohort=cohort)
    )
    pooled_summary_rows.append(pooled_regime_summary)

PLOT_DF = pd.concat(pooled_agg_rows, ignore_index=True)
REGIME_SUMMARY_DF = pd.concat(pooled_summary_rows, ignore_index=True)

display(
    REGIME_SUMMARY_DF.assign(regime=lambda df: df['regime'].map(REGIME_PRETTY_NAMES))
    [['cohort', 'regime', 'mean_ratio_over_k', 'median_ratio_over_k', 'mean_normalized_auc_ratio']]
    .sort_values(['cohort', 'mean_ratio_over_k'], ascending=[True, False])
    .reset_index(drop=True)
    .round(4)
)


In [ ]:
fig, axes = plt.subplots(
    1,
    len(COHORTS),
    figsize=(7.2, 3.0),
    sharey=True,
    gridspec_kw={'wspace': 0.08},
)

if len(COHORTS) == 1:
    axes = [axes]

panel_labels = ['a', 'b', 'c', 'd']

for ax, panel_label, (cohort, meta) in zip(axes, panel_labels, COHORTS.items()):
    cohort_df = PLOT_DF[PLOT_DF['cohort'] == cohort].copy()

    for regime in REGIME_ORDER:
        curve_df = cohort_df[cohort_df['regime'] == regime].sort_values('k')
        ax.plot(
            curve_df['k'],
            curve_df['global_selected_over_annotated_ratio'],
            color=REGIME_COLORS[regime],
            linestyle=REGIME_LINESTYLES[regime],
            marker=REGIME_MARKERS[regime],
            markerfacecolor='white',
            markeredgewidth=0.8,
            linewidth=1.4 if regime == 'reveal_one_by_one_yhat_round_robin_prob_update' else 1.1,
            alpha=1.0 if regime == 'reveal_one_by_one_yhat_round_robin_prob_update' else 0.9,
            zorder=5 if regime == 'reveal_one_by_one_yhat_round_robin_prob_update' else 3,
            label=REGIME_PRETTY_NAMES[regime],
        )

    ax.axhline(1.0, color='black', linestyle=':', linewidth=0.5, alpha=0.4)
    ax.text(20.6, 1.0, 'opt.', fontsize=5.5, va='center', color='#555555')

    ax.set_xlim(0.5, 20.5)
    ax.set_ylim(0.35, 1.05)
    ax.set_xlabel('Exemplars per class (m)')
    ax.set_xticks([1, 5, 10, 15, 20])
    ax.grid(axis='y', linewidth=0.3, alpha=0.5)
    ax.set_title(meta['title'], pad=8)
    ax.text(
        -0.08,
        1.04,
        panel_label,
        transform=ax.transAxes,
        fontsize=12,
        fontweight='bold',
        va='bottom',
        ha='left',
    )

axes[0].set_ylabel(r'Selection efficiency ratio ($\eta$)')
for i, ax in enumerate(axes):
    if i == 0:
        ax.legend(
            loc='lower right',
            frameon=True,
            fancybox=False,
            edgecolor='#cccccc',
            framealpha=0.95,
            borderpad=0.4,
        )
    else:
        ax.legend(
            loc='center right',
            frameon=True,
            fancybox=False,
            edgecolor='#cccccc',
            framealpha=0.95,
            borderpad=0.4,
        )

plt.show()


In [ ]:
output_dir = REPO_ROOT / 'notebooks'
output_stem = output_dir / 'fig_selection_efficiency'

fig.savefig(output_stem.with_suffix('.pdf'))
fig.savefig(output_stem.with_suffix('.png'))
print(f'Saved: {output_stem.with_suffix(".pdf")} and {output_stem.with_suffix(".png")}')
